# Phase 2A: MedGemma 4B Setup and Validation

**Project**: An Explainable Multi-Agent Clinical Intelligence Platform for Clinical Decision Assistance  
**Layer**: Claim Verification Agent (Local LLM Setup & Hardware Validation)  
**Target Model**: `google/medgemma-4b-it`  

> **Phase 1 Integrity Guard**:  
> Phase 1 (Evidence Retrieval Agent, BioASQ evidence corpus `bioasq_evidence_corpus_dedup.csv`, embeddings `bioasq_embeddings_dedup.npy`, FAISS index `bioasq_faiss_dedup.index`, provenance lookup `evidence_lookup_dedup.csv`, and retrieval evaluation `retrieval_eval_results.csv`) is 100% frozen, complete, and untouched.

---

## Step 1: Detect Hardware & System Environment

Before attempting model execution, inspect and report:
- Python version
- PyTorch version
- CUDA availability & version
- GPU device name
- Total & available GPU VRAM
- bfloat16 hardware support

**Safety Rule**: If CUDA is unavailable, stop execution immediately. Do not silently fall back to CPU for the 4B model.

In [1]:
import sys
import torch

print("=" * 60)
print("1. HARDWARE & SYSTEM ENVIRONMENT DETECTION")
print("=" * 60)
python_ver = sys.version.split()[0]
pytorch_ver = torch.__version__
cuda_avail = torch.cuda.is_available()

print(f"Python:             {python_ver}")
print(f"PyTorch:            {pytorch_ver}")
print(f"CUDA available:     {cuda_avail}")

if not cuda_avail:
    print("\n[CRITICAL ERROR] CUDA is NOT available on this system.")
    print("Execution halted. GPU execution is strictly required for MedGemma 4B.")
    raise SystemExit("CUDA is unavailable. Halting execution per project protocol.")

cuda_ver = torch.version.cuda
gpu_name = torch.cuda.get_device_name(0)
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
free_vram_gb = torch.cuda.mem_get_info()[0] / (1024**3)
bf16_supported = torch.cuda.is_bf16_supported()

print(f"CUDA version:       {cuda_ver}")
print(f"GPU:                {gpu_name}")
print(f"GPU VRAM Total:     {total_vram_gb:.2f} GB")
print(f"GPU VRAM Available: {free_vram_gb:.2f} GB")
print(f"bfloat16 supported: {bf16_supported}")
print("=" * 60)

1. HARDWARE & SYSTEM ENVIRONMENT DETECTION
Python:             3.10.11
PyTorch:            2.6.0+cu124
CUDA available:     True
CUDA version:       12.4
GPU:                NVIDIA GeForce RTX 2050
GPU VRAM Total:     4.00 GB
GPU VRAM Available: 3.22 GB
bfloat16 supported: True


## Step 2: Check Required Packages

Verify installation of core stack packages:
- `torch`
- `transformers`
- `accelerate`
- `huggingface_hub`

In [2]:
import torch
import transformers
import accelerate
import huggingface_hub

print("=" * 60)
print("2. REQUIRED PACKAGES CHECK")
print("=" * 60)
print(f"torch:           {torch.__version__}")
print(f"transformers:    {transformers.__version__}")
print(f"accelerate:      {accelerate.__version__}")
print(f"huggingface_hub: {huggingface_hub.__version__}")
print("All required packages verified successfully.")
print("=" * 60)

2. REQUIRED PACKAGES CHECK
torch:           2.6.0+cu124
transformers:    5.18.0
accelerate:      1.15.0
huggingface_hub: 1.33.0
All required packages verified successfully.


## Step 3: Hugging Face Authentication Check

`google/medgemma-4b-it` is a gated Hugging Face repository requiring accepted terms and authentication.

> **Security Notice**: Never hardcode, print, or commit access tokens to notebooks or git repositories.

In [3]:
from huggingface_hub import HfApi, get_token

print("=" * 60)
print("3. HUGGING FACE AUTHENTICATION CHECK")
print("=" * 60)
token = get_token()
if not token:
    print("[AUTHENTICATION REQUIRED] No Hugging Face access token found.")
    print("\nPlease complete the following steps:")
    print("1. Open: https://huggingface.co/google/medgemma-4b-it and accept Google's usage terms.")
    print("2. Create an access token at: https://huggingface.co/settings/tokens")
    print("3. In terminal, run: hf auth login (or huggingface-cli login)")
    print("   Or interactively: from huggingface_hub import login; login()")
    raise PermissionError("Missing Hugging Face authentication token.")

api = HfApi()
user_info = api.whoami()
username = user_info.get("name") or user_info.get("fullname", "Authenticated User")
print(f"Authentication Status: Logged in as '{username}'")
print(f"User Type:             {user_info.get('type', 'user')}")
print("Hugging Face credentials verified.")
print("=" * 60)

3. HUGGING FACE AUTHENTICATION CHECK
Authentication Status: Logged in as 'Bharath129'
User Type:             user
Hugging Face credentials verified.


## Step 4: Load MedGemma 4B Model & Tokenizer

- Model ID: `google/medgemma-4b-it`
- Precision: Safely auto-detected (`torch.bfloat16` if supported by GPU, else `torch.float16`)
- Device allocation: `device_map="auto"`

### VRAM Safety & Error Handling
Because development hardware utilizes an NVIDIA GeForce RTX 2050 (4.0 GB VRAM):
- `google/medgemma-4b-it` weights occupy ~8.60 GB in 16-bit precision.
- Total dedicated GPU VRAM (4.0 GB) is insufficient to hold the full unquantized weights.
- If loading causes CUDA OOM or memory faults, the safety protocol captures the error, reports available VRAM, and halts further model execution without crashing the workflow or substituting models.

In [4]:
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "google/medgemma-4b-it"

# Determine safe precision based on actual GPU capabilities
if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    chosen_dtype = torch.bfloat16
else:
    chosen_dtype = torch.float16

print("=" * 60)
print("4. LOADING TOKENIZER & MEDGEMMA 4B MODEL")
print("=" * 60)
print(f"Target Model:       {MODEL_ID}")
print(f"Selected Precision: {chosen_dtype}")
print(f"Device Map:         auto")

# 1. Load Tokenizer
print("\n[1/2] Loading Tokenizer...")
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    tokenizer_status = "Loaded Successfully"
    print(f"Tokenizer Status:   {tokenizer_status}")
    print(f"Vocab Size:         {tokenizer.vocab_size:,}")
except Exception as e:
    tokenizer_status = f"Failed: {e}"
    print(f"Tokenizer Error:    {e}")
    tokenizer = None

# 2. Load Model with VRAM Safety Protocol
model = None
model_loaded = False
vram_issue_reported = False
vram_error_message = None

if tokenizer is not None:
    print(f"\n[2/2] Checking VRAM Capacity for '{MODEL_ID}'...")
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    free_vram_gb = torch.cuda.mem_get_info()[0] / (1024**3)
    REQUIRED_VRAM_GB = 8.0
    
    if total_vram_gb < REQUIRED_VRAM_GB:
        vram_issue_reported = True
        vram_error_message = (
            f"VRAM limit reached: GPU possesses {total_vram_gb:.2f} GB VRAM, but unquantized "
            f"{MODEL_ID} requires ~{REQUIRED_VRAM_GB:.1f} GB VRAM. "
            f"Full local unquantized execution is not feasible on this device."
        )
        print("\n" + "!" * 60)
        print("[VRAM SAFETY PROTOCOL ACTIVATED]")
        print(f"Hardware:           {gpu_name}")
        print(f"GPU Total VRAM:     {total_vram_gb:.2f} GB")
        print(f"GPU Available VRAM: {free_vram_gb:.2f} GB")
        print(f"Model Footprint:    ~8.60 GB (unquantized {chosen_dtype})")
        print(f"Diagnostic Report:  {vram_error_message}")
        print("\nFull local execution is not feasible under the current configuration.")
        print("STOPPING Phase 2A per safety protocol. No unauthorized models substituted.")
        print("!" * 60)
        model_loaded = False
    else:
        try:
            model = AutoModelForCausalLM.from_pretrained(
                MODEL_ID,
                dtype=chosen_dtype,
                device_map="auto",
                low_cpu_mem_usage=True
            )
            model_loaded = True
            param_count = sum(p.numel() for p in model.parameters())
            active_device = next(model.parameters()).device
            
            print(f"Model Loading:      SUCCESS")
            print(f"Model ID:           {MODEL_ID}")
            print(f"Parameter Count:    {param_count:,}")
            print(f"Primary Device:     {active_device}")
            print(f"Loaded Dtype:       {model.dtype}")
            print(f"Tokenizer Status:   {tokenizer_status}")
            
        except (torch.cuda.OutOfMemoryError, RuntimeError, Exception) as e:
            vram_issue_reported = True
            vram_error_message = str(e)
            model_loaded = False
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            print(f"Model load exception: {e}")

print("=" * 60)

4. LOADING TOKENIZER & MEDGEMMA 4B MODEL
Target Model:       google/medgemma-4b-it
Selected Precision: torch.bfloat16
Device Map:         auto

[1/2] Loading Tokenizer...
Tokenizer Status:   Loaded Successfully
Vocab Size:         262,144

[2/2] Checking VRAM Capacity for 'google/medgemma-4b-it'...

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
[VRAM SAFETY PROTOCOL ACTIVATED]
Hardware:           NVIDIA GeForce RTX 2050
GPU Total VRAM:     4.00 GB
GPU Available VRAM: 3.22 GB
Model Footprint:    ~8.60 GB (unquantized torch.bfloat16)
Diagnostic Report:  VRAM limit reached: GPU possesses 4.00 GB VRAM, but unquantized google/medgemma-4b-it requires ~8.0 GB VRAM. Full local unquantized execution is not feasible on this device.

Full local execution is not feasible under the current configuration.
STOPPING Phase 2A per safety protocol. No unauthorized models substituted.
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!


## Step 5: Basic Medical Generation Test

**Objective**: Perform a lightweight, deterministic generation test with MedGemma 4B.
- **Prompt**: `"What is Hirschsprung disease? Give a concise medical explanation."`
- **Configuration**: `do_sample=False`, bounded token length.
- **Scope**: Integration verification only (not clinical truth).

In [5]:
print("=" * 60)
print("5. BASIC MEDICAL GENERATION TEST")
print("=" * 60)
basic_test_passed = False
basic_prompt = "What is Hirschsprung disease? Give a concise medical explanation."

if model_loaded and model is not None and tokenizer is not None:
    try:
        messages = [{"role": "user", "content": basic_prompt}]
        prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=150,
                do_sample=False
            )
            
        input_len = inputs["input_ids"].shape[1]
        output_text = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
        
        print("MODEL TEST")
        print("----------")
        print(f"Input:\n{basic_prompt}\n")
        print(f"Output:\n{output_text}\n")
        basic_test_passed = True
        print("[STATUS] Basic Generation Test: PASS")
    except Exception as e:
        print(f"[STATUS] Basic Generation Test Failed: {e}")
else:
    print("[BYPASSED] Model not resident in VRAM due to hardware constraints.")
    print("[STATUS] Basic Generation Test: SKIPPED (VRAM limit)")
print("=" * 60)

5. BASIC MEDICAL GENERATION TEST
[BYPASSED] Model not resident in VRAM due to hardware constraints.
[STATUS] Basic Generation Test: SKIPPED (VRAM limit)


## Step 6: Evidence-Grounded Claim Verification Test

**Objective**: Perform ONE controlled evidence-grounded claim verification test.
- **Claim**: `"RET mutations are associated with Hirschsprung disease."`
- **Evidence ID**: `E000008` (from `datasets/processed/bioasq/evidence_lookup_dedup.csv`)
- **Allowed labels**: `SUPPORTED`, `CONTRADICTED`, `INSUFFICIENT`
- **Expected Outcome**: `SUPPORTED`

In [6]:
import pandas as pd
from pathlib import Path

print("=" * 60)
print("6. EVIDENCE-GROUNDED CLAIM VERIFICATION TEST")
print("=" * 60)

# Resolve path to Phase 1 evidence lookup
lookup_file = Path("../datasets/processed/bioasq/evidence_lookup_dedup.csv")
if not lookup_file.exists():
    lookup_file = Path("datasets/processed/bioasq/evidence_lookup_dedup.csv")

df_lookup = pd.read_csv(lookup_file)
ev_row = df_lookup[df_lookup["evidence_id"] == "E000008"]

if ev_row.empty:
    raise FileNotFoundError("Evidence ID E000008 not found in lookup corpus.")

evidence_id = "E000008"
evidence_text = ev_row.iloc[0]["snippet_text"]
claim_test_1 = "RET mutations are associated with Hirschsprung disease."

prompt_grounded = f"""You are a clinical claim verification assistant.

Your task is to determine whether the CLAIM is supported by the PROVIDED EVIDENCE.

You must use only the provided evidence.
Do not rely on outside medical knowledge.

Allowed labels:
SUPPORTED
CONTRADICTED
INSUFFICIENT

CLAIM:
{claim_test_1}

EVIDENCE:
[{evidence_id}]
{evidence_text}

Return:

Label:
Evidence IDs:
Rationale:"""

print("Configured Prompt:")
print(prompt_grounded)
print("-" * 60)

evidence_grounded_passed = False
evidence_grounded_output = ""

if model_loaded and model is not None and tokenizer is not None:
    try:
        messages = [{"role": "user", "content": prompt_grounded}]
        prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=150,
                do_sample=False
            )
            
        input_len = inputs["input_ids"].shape[1]
        evidence_grounded_output = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
        
        print("MODEL TEST (Evidence Grounded)")
        print("------------------------------")
        print(evidence_grounded_output)
        print("------------------------------")
        
        if "SUPPORTED" in evidence_grounded_output.upper():
            evidence_grounded_passed = True
            print("[STATUS] Evidence-Grounded Test: PASS")
        else:
            print("[STATUS] Evidence-Grounded Test: FAIL (Expected SUPPORTED label)")
    except Exception as e:
        print(f"[STATUS] Evidence-Grounded Test Failed: {e}")
else:
    print("[BYPASSED] Model not resident in VRAM due to hardware constraints.")
    print("[STATUS] Evidence-Grounded Test: SKIPPED (VRAM limit)")
print("=" * 60)

6. EVIDENCE-GROUNDED CLAIM VERIFICATION TEST
Configured Prompt:
You are a clinical claim verification assistant.

Your task is to determine whether the CLAIM is supported by the PROVIDED EVIDENCE.

You must use only the provided evidence.
Do not rely on outside medical knowledge.

Allowed labels:
SUPPORTED
CONTRADICTED
INSUFFICIENT

CLAIM:
RET mutations are associated with Hirschsprung disease.

EVIDENCE:
[E000008]
In the etiology of Hirschsprung disease various genes play a role; these are: RET, EDNRB, GDNF, EDN3 and SOX10, NTN3, ECE1, Mutations in these genes may result in dominant, recessive or multifactorial patterns of inheritance.

Return:

Label:
Evidence IDs:
Rationale:
------------------------------------------------------------
[BYPASSED] Model not resident in VRAM due to hardware constraints.
[STATUS] Evidence-Grounded Test: SKIPPED (VRAM limit)


## Step 7: Insufficient Evidence Test (Abstention Verification)

**Objective**: Test abstention behavior when the claim cannot be established from the evidence.
- **Claim**: `"Amoxicillin is recommended as the first-line curative pharmacotherapy for Hirschsprung disease."`
- **Evidence**: `E000008` (Pertains strictly to etiology/genetics, without antibiotic data)
- **Expected Label**: `INSUFFICIENT`
- **Rule**: Do not force the model to hallucinate SUPPORTED or CONTRADICTED.

In [7]:
print("=" * 60)
print("7. INSUFFICIENT EVIDENCE TEST (ABSTENTION)")
print("=" * 60)

claim_test_2 = "Amoxicillin is recommended as the first-line curative pharmacotherapy for Hirschsprung disease."

prompt_insufficient = f"""You are a clinical claim verification assistant.

Your task is to determine whether the CLAIM is supported by the PROVIDED EVIDENCE.

You must use only the provided evidence.
Do not rely on outside medical knowledge.

Allowed labels:
SUPPORTED
CONTRADICTED
INSUFFICIENT

CLAIM:
{claim_test_2}

EVIDENCE:
[{evidence_id}]
{evidence_text}

Return:

Label:
Evidence IDs:
Rationale:"""

insufficient_passed = False
insufficient_output = ""

if model_loaded and model is not None and tokenizer is not None:
    try:
        messages = [{"role": "user", "content": prompt_insufficient}]
        prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=150,
                do_sample=False
            )
            
        input_len = inputs["input_ids"].shape[1]
        insufficient_output = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
        
        print("MODEL TEST (Abstention / Insufficient Evidence)")
        print("-----------------------------------------------")
        print(insufficient_output)
        print("-----------------------------------------------")
        
        if "INSUFFICIENT" in insufficient_output.upper():
            insufficient_passed = True
            print("[STATUS] Insufficient-Evidence Test: PASS")
        else:
            print("[STATUS] Insufficient-Evidence Test: FAIL (Expected INSUFFICIENT label)")
    except Exception as e:
        print(f"[STATUS] Insufficient-Evidence Test Failed: {e}")
else:
    print("[BYPASSED] Model not resident in VRAM due to hardware constraints.")
    print("[STATUS] Insufficient-Evidence Test: SKIPPED (VRAM limit)")
print("=" * 60)

7. INSUFFICIENT EVIDENCE TEST (ABSTENTION)
[BYPASSED] Model not resident in VRAM due to hardware constraints.
[STATUS] Insufficient-Evidence Test: SKIPPED (VRAM limit)


## Step 8: Structured Output Parser Test

**Objective**: Validate parsing of model responses into the canonical schema:
- `Label`: restricted strictly to `SUPPORTED`, `CONTRADICTED`, `INSUFFICIENT`
- `Evidence IDs`: List of cited evidence IDs
- `Rationale`: Clinical justification extracted from text

Tests both benchmark exemplar formatting and live model output (if available).

In [8]:
import re

print("=" * 60)
print("8. STRUCTURED OUTPUT PARSER & VALIDATION")
print("=" * 60)

def parse_clinical_verification(text: str) -> dict:
    """
    Parses structured claim verification text.
    Allowed labels ONLY: SUPPORTED, CONTRADICTED, INSUFFICIENT.
    """
    parsed = {
        "label": None,
        "evidence_ids": [],
        "rationale": None,
        "valid_format": False
    }
    
    if not text:
        return parsed
        
    # 1. Parse Label
    lbl_match = re.search(r"Label:\s*(SUPPORTED|CONTRADICTED|INSUFFICIENT)", text, re.IGNORECASE)
    if lbl_match:
        parsed["label"] = lbl_match.group(1).upper()
        
    # 2. Parse Evidence IDs
    ev_match = re.search(r"Evidence IDs?:\s*([^\n\r]+)", text, re.IGNORECASE)
    if ev_match:
        raw = ev_match.group(1).strip()
        found_ids = re.findall(r"E\d+", raw)
        parsed["evidence_ids"] = found_ids if found_ids else [raw]
        
    # 3. Parse Rationale
    rat_match = re.search(r"Rationale:\s*([\s\S]+)", text, re.IGNORECASE)
    if rat_match:
        parsed["rationale"] = rat_match.group(1).strip()
        
    if parsed["label"] in ["SUPPORTED", "CONTRADICTED", "INSUFFICIENT"] and parsed["rationale"]:
        parsed["valid_format"] = True
        
    return parsed

# 1. Benchmark exemplar test
canonical_sample = f"""Label: SUPPORTED
Evidence IDs: {evidence_id}
Rationale: The provided snippet explicitly documents that RET mutations contribute to the etiology and risk of Hirschsprung disease."""

sample_res = parse_clinical_verification(canonical_sample)
print("Benchmark Exemplar Parsing Test:")
print(f"  Parsed Result: {sample_res}")
structured_test_passed = sample_res["valid_format"]
print(f"  Benchmark Parsing Status: {'PASS' if structured_test_passed else 'FAIL'}")

# 2. Live model response parsing test (if model generated response)
if evidence_grounded_output:
    print("\nLive Model Response Parsing Test:")
    live_res = parse_clinical_verification(evidence_grounded_output)
    print(f"  Parsed Result: {live_res}")
    print(f"  Live Parsing Status: {'PASS' if live_res['valid_format'] else 'FAIL'}")

print("=" * 60)

8. STRUCTURED OUTPUT PARSER & VALIDATION
Benchmark Exemplar Parsing Test:
  Parsed Result: {'label': 'SUPPORTED', 'evidence_ids': ['E000008'], 'rationale': 'The provided snippet explicitly documents that RET mutations contribute to the etiology and risk of Hirschsprung disease.', 'valid_format': True}
  Benchmark Parsing Status: PASS


## Step 9: Phase 2A Diagnostic & Validation Report

Comprehensive summary answering items A through I per Phase 2A specification.

In [9]:
print("=" * 60)
print("PHASE 2A FINAL VALIDATION REPORT")
print("=" * 60)
print("A. Environment:")
print(f"   - Python:              {python_ver}")
print(f"   - PyTorch:             {pytorch_ver}")
print(f"   - CUDA Available:      {cuda_avail} (CUDA {cuda_ver})")
print(f"   - GPU:                 {gpu_name}")
print(f"   - Total VRAM:          {total_vram_gb:.2f} GB")
print(f"   - Free VRAM:           {free_vram_gb:.2f} GB")

print("\nB. Packages:")
print(f"   - transformers:        {transformers.__version__}")
print(f"   - accelerate:          {accelerate.__version__}")
print(f"   - huggingface_hub:     {huggingface_hub.__version__}")

print("\nC. Model:")
print(f"   - Exact Model ID:      {MODEL_ID}")
print(f"   - Loading Status:      {'SUCCESS' if model_loaded else 'VRAM CONSTRAINED'}")
print(f"   - Selected Dtype:      {chosen_dtype}")
print(f"   - Device Map:          {'cuda' if model_loaded else 'auto (exceeds VRAM)'}")

print("\nD. Basic Generation Test:")
print(f"   - Status:              {'PASS' if basic_test_passed else ('SKIPPED (VRAM limit)' if not model_loaded else 'FAIL')}")

print("\nE. Evidence-Grounded Test:")
print(f"   - Status:              {'PASS' if evidence_grounded_passed else ('SKIPPED (VRAM limit)' if not model_loaded else 'FAIL')}")

print("\nF. Insufficient-Evidence Test:")
print(f"   - Status:              {'PASS' if insufficient_passed else ('SKIPPED (VRAM limit)' if not model_loaded else 'FAIL')}")

print("\nG. Structured Output Test:")
print(f"   - Status:              {'PASS' if structured_test_passed else 'FAIL'}")

print("\nH. VRAM / Runtime Issues:")
if vram_issue_reported or not model_loaded:
    print(f"   - Hardware Constraint: RTX 2050 possesses 4.0 GB VRAM.")
    print(f"     Full local execution of unquantized 4B parameters (~8.60 GB weights)")
    print(f"     exceeds dedicated GPU VRAM capacity.")
    if vram_error_message:
        print(f"     Diagnostic Message:  {vram_error_message}")
else:
    print("   - None. Execution completed within available GPU VRAM.")

print("\nI. Exact Notebook Created:")
print("   - notebook/03_claim_verification.ipynb")
print("=" * 60)

PHASE 2A FINAL VALIDATION REPORT
A. Environment:
   - Python:              3.10.11
   - PyTorch:             2.6.0+cu124
   - CUDA Available:      True (CUDA 12.4)
   - GPU:                 NVIDIA GeForce RTX 2050
   - Total VRAM:          4.00 GB
   - Free VRAM:           3.22 GB

B. Packages:
   - transformers:        5.18.0
   - accelerate:          1.15.0
   - huggingface_hub:     1.33.0

C. Model:
   - Exact Model ID:      google/medgemma-4b-it
   - Loading Status:      VRAM CONSTRAINED
   - Selected Dtype:      torch.bfloat16
   - Device Map:          auto (exceeds VRAM)

D. Basic Generation Test:
   - Status:              SKIPPED (VRAM limit)

E. Evidence-Grounded Test:
   - Status:              SKIPPED (VRAM limit)

F. Insufficient-Evidence Test:
   - Status:              SKIPPED (VRAM limit)

G. Structured Output Test:
   - Status:              PASS

H. VRAM / Runtime Issues:
   - Hardware Constraint: RTX 2050 possesses 4.0 GB VRAM.
     Full local execution of unquantized 4B 